# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirl0w/Machine-Learning-Intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Finding 1: [paste exact claim from the paper here]
My methodology question: [where does this label/measurement come from? was
there a control group or holdout? could an alternate explanation — seasonality,
consolidation — produce the same pattern?]

Finding 2: [paste exact claim from the paper here]
My methodology question: [same style — respectful, concrete, specific to this claim]

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X, y, test_size=0.2, random_state=42)
tree_n = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_n.fit(X_train_n, y_train_n)
naive_p50 = precision_at_k(tree_n.predict_proba(X_test_n)[:,1], y_test_n.values, 50)

# AFTER: honest client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
tree_g = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_g.fit(X_train_g, y_train_g)
grouped_p50 = precision_at_k(tree_g.predict_proba(X_test_g)[:,1], y_test_g.values, 50)

print(f"BEFORE (naive split)   Precision@50: {naive_p50:.3f}")
print(f"AFTER  (client-holdout) Precision@50: {grouped_p50:.3f}")

BEFORE (naive split)   Precision@50: 0.660
AFTER  (client-holdout) Precision@50: 0.600


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
print("Leakage checklist:")
print("- Any post-decision features?", "No — all features observable at prediction time")
print("- Feature/target window overlap?", "No — trend_direction excluded as a feature")
print("- Product decision flags used?", "No — not present in starter data")
print("- Train/test row independence?", "Fixed via client-grouped split in Section 2")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Original claim (from Week 5 notebook): "[paste your boldest sentence from
w05_model.ipynb]"

Rewritten: "Under a client-holdout validation split, the model showed a
[X]% Precision@50 versus the baseline's [Y]% — an observed, directional
improvement on this dataset. This is decision-support evidence, not a
guarantee on unseen clients or future periods."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.